# Phân tích dataset — RSNA Knee MRI

Cấu trúc, độ phủ nhãn, ngôn ngữ của report, chuẩn hóa cường độ và ảnh mẫu.
Chạy được cả local lẫn Colab.


In [ ]:
# 0. Bootstrap: lấy code, cài package, nạp cấu hình — chạy được cả Colab lẫn local.
#    Code luôn được clone/pull vào /content (Colab) rồi `pip install -e`, nên không còn
#    phụ thuộc vào việc bạn đã tự chép src/ lên Drive. Chỉ dataset/, artifacts/ và
#    vendor/ nằm trên Drive (bền vững giữa các lần runtime bị ngắt).
import logging, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/hoangtung386/Att-kneenet.git'
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/Challenge_RNSA_KNEE')
    CODE = Path('/content/Att-kneenet')
    if (CODE / '.git').is_dir():
        subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CODE)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[viz,eval]'], check=True)
else:
    CODE = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
    DRIVE_ROOT = CODE
sys.path.insert(0, str(CODE / 'src'))

assert (DRIVE_ROOT / 'dataset' / 'train.csv').is_file(), (
    f'Không thấy dataset tại {DRIVE_ROOT / "dataset"} — hãy đặt dataset RSNA vào đó.'
)
print('code   :', CODE)
print('dataset:', DRIVE_ROOT / 'dataset')

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

from knee_mri.constants import COL_FAT, COL_FLUID, COL_PLANE, LABELS
from knee_mri.data.catalog import StudyCatalog

# 1. Nạp cấu hình
from knee_mri.config import load_config
from knee_mri.utils.logging import setup_logging

RUN = 'full'   # 'full' | 'pilot' — phải trùng với RUN ở notebook 02
ARTIFACTS = DRIVE_ROOT / 'artifacts' / ('pilot' if RUN == 'pilot' else '')
cfg = load_config('colab' if IN_COLAB else None, overrides={
    'paths': {
        'data_root': str(DRIVE_ROOT / 'dataset'),
        'artifacts_root': str(ARTIFACTS),
        'dino_code_dir': str(DRIVE_ROOT / 'vendor' / '3DINO'),
    },
})
setup_logging(logging.WARNING)
print('môi trường:', cfg.env, '| dataset:', cfg.paths.data_root, '| artifacts:', cfg.paths.artifacts_root)

## 1. Quy mô và độ phủ nhãn

In [ ]:
catalog = StudyCatalog.from_csv(cfg.paths.train_csv, cfg.paths.train_series_csv)
series_df = pd.read_csv(cfg.paths.train_series_csv)

gold_uids = catalog.studies_with_gold()
print('Study (train)        :', len(catalog))
print('Series (train)       :', len(series_df))
print('Series / study (tb)  : %.2f' % (len(series_df) / max(1, len(catalog))))
print('Study có report      :', sum(1 for u in catalog.studies() if catalog.report(u).strip()))
print('Study có nhãn gold   :', len(gold_uids))

_, gold_rows = catalog.gold_matrix(gold_uids)
positives = np.array(gold_rows).sum(axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(LABELS, positives, color='#c44e52')
ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels(LABELS, rotation=45, ha='right')
ax.set_ylabel('số study dương tính')
ax.set_title(f'Nhãn gold rất thưa (chỉ {len(gold_uids)}/{len(catalog)} study được gán)')
fig.tight_layout()
plt.show()

## 2. Ngôn ngữ của report

Đây là lý do bộ gán nhãn theo luật phải xử lý phủ định đa ngôn ngữ: bản trước
refactor chỉ nhận phủ định tiếng Anh nên gán dương tính sai cho hàng trăm study.


In [ ]:
LANGUAGE_CUES = {
    'Anh':          ['knee', 'no evidence', 'without', 'effusion'],
    'Tây Ban Nha':  ['rodilla', 'menisco', 'derrame', 'sin '],
    'Hà Lan':       ['knie', 'geen', 'meniscus', 'kruisband'],
    'Đức':          ['kniegelenk', 'kein', 'erguss', 'ohne'],
    'Pháp':         ['genou', 'sans ', 'pas de'],
}

counts = Counter()
for study_uid in catalog.studies():
    text = catalog.report(study_uid).lower()
    best, best_score = 'không rõ', 0
    for language, cues in LANGUAGE_CUES.items():
        score = sum(cue in text for cue in cues)
        if score > best_score:
            best, best_score = language, score
    counts[best] += 1

for language, count in counts.most_common():
    print(f'  {language:<14s} {count:>5d}')

fig, ax = plt.subplots(figsize=(7, 4))
labels_, values_ = zip(*counts.most_common())
ax.bar(labels_, values_, color='#8172b2')
ax.set_ylabel('số study')
ax.set_title('Ngôn ngữ của report (ước lượng)')
fig.tight_layout()
plt.show()

## 3. Chất lượng weak label so với nhãn gold

In [ ]:
from knee_mri.labeling import format_labeler_report, keyword_labeler, score_labeler

# Chấm luật từ khóa trên các study có nhãn gold, theo TỪNG nhãn: con số gộp che mất
# nhãn yếu nhất (thường là OA/Synovitis/PF OA khi report không thuộc ngôn ngữ trong từ điển).
keyword_report = score_labeler(catalog, lambda uid: keyword_labeler(catalog.report(uid)))
print(format_labeler_report(keyword_report))

# Medial và Lateral Meniscus trùng nhau nhiều không? Tự nó không phải lỗi: nhiều study
# bình thường hoặc tổn thương cả hai bên. So với chính nhãn gold mới có ý nghĩa.
def coupled(get):
    return sum(get(u)['Medial Meniscus'] == get(u)['Lateral Meniscus'] for u in gold_uids)
rule_coupled = coupled(lambda u: keyword_labeler(catalog.report(u)))
print(f"\nMedial == Lateral Meniscus: luật {rule_coupled}/{len(gold_uids)}"
      f" | gold thật {coupled(catalog.gold)}/{len(gold_uids)}")
print('(bản cũ dùng chung từ khóa "menisc" nên luật luôn trùng 100%)')

## 4. Phân bố metadata series

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, column in zip(axes, [COL_PLANE, COL_FLUID, COL_FAT]):
    series_df[column].value_counts().plot(kind='bar', ax=ax, color='#4c72b0')
    ax.set_title(column)
    ax.tick_params(axis='x', rotation=0)
fig.tight_layout()
plt.show()

## 5. Ảnh mẫu dưới các chế độ chuẩn hóa

In [ ]:
from knee_mri.data.dicom_io import read_series
from knee_mri.data.normalize import NORM_MODES, normalize, to_unit_range
from knee_mri.data.series_selection import representative_series

study_uid = catalog.studies_with_series()[0]
series = representative_series(catalog, study_uid, cfg.data)
series_dir = cfg.paths.series_dir(study_uid, series.series_uid)
raw = read_series(series_dir)
print('shape thô:', raw.shape, '| dải giá trị:', raw.min(), '→', raw.max())

middle = raw.shape[0] // 2
modes = [m for m in NORM_MODES if m != 'none']
fig, axes = plt.subplots(1, len(modes), figsize=(4 * len(modes), 4))
for ax, mode in zip(axes, modes):
    plane = normalize(raw, mode)[middle]
    ax.imshow(to_unit_range(plane, mode), cmap='gray')
    ax.set_title(mode + ('  ← đang dùng' if mode == cfg.data.norm_mode else ''), fontsize=10)
    ax.axis('off')
    
fig.suptitle('Cùng một lát cắt dưới các chế độ chuẩn hóa')
fig.tight_layout()
plt.show()

In [ ]:
# 6. Volume đã dựng: chuỗi lát ở shape cố định mà model thực sự nhìn thấy
from knee_mri.data.volume import build_volume

volume = build_volume(series_dir, cfg.data)
print('shape sau build_volume:', volume.shape, '(luôn bằng cfg.data.target_shape)')

indices = np.linspace(0, volume.shape[0] - 1, min(8, volume.shape[0])).astype(int)
fig, axes = plt.subplots(1, len(indices), figsize=(2.2 * len(indices), 3))
for ax, index in zip(np.atleast_1d(axes), indices):
    ax.imshow(to_unit_range(volume[index], cfg.data.norm_mode), cmap='gray')
    ax.set_title(f'lát {index}', fontsize=9)
    ax.axis('off')
fig.tight_layout()
plt.show()

## Kết luận

- Nhãn cấu trúc cực thưa (58/4407) → weak label từ report là bắt buộc, và 58
  study đó nên dành cho validation chứ không phải huấn luyện.
- Report đa ngôn ngữ → xử lý phủ định phải phủ ít nhất Anh/Tây Ban Nha/Hà
  Lan/Đức/Pháp.
- Chuẩn hóa: chọn `volume_percentile_pm1` — xem
  [`docs/normalization.md`](../docs/normalization.md) để biết lý do đầy đủ.
